In [1]:
from __future__ import annotations

import os
from getpass import getpass

import weaviate
import weaviate.classes as wvc

from weaviate.classes.init import Auth
from weaviate.classes.query import Filter, MetadataQuery

In [2]:
if "WEAVIATE_URL" not in os.environ:
    os.environ["WEAVIATE_URL"] = getpass("WEAVIATE_URL: ")

if "WEAVIATE_API_KEY" not in os.environ:
    os.environ["WEAVIATE_API_KEY"] = getpass("WEAVIATE_API_KEY: ")

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

In [3]:
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.environ["WEAVIATE_URL"],
    auth_credentials=Auth.api_key(os.environ["WEAVIATE_API_KEY"]),
    headers={
        "X-OpenAI-Api-Key": os.environ["OPENAI_API_KEY"],
    },
)

print("Client ready:", client.is_ready())

Client ready: True


In [4]:
COLLECTION_NAME = "LearningArticle"

if client.collections.exists(COLLECTION_NAME):
    client.collections.delete(COLLECTION_NAME)

articles = client.collections.create(
    name=COLLECTION_NAME,
    vector_config=wvc.config.Configure.Vectors.text2vec_openai(
        model="text-embedding-3-small",
    ),
    generative_config=wvc.config.Configure.Generative.openai(
        model="gpt-4o-mini",
    ),
    properties=[
        wvc.config.Property(
            name="title",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="content",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="source",
            data_type=wvc.config.DataType.TEXT,
        ),
    ],
)

print("Created collection:", COLLECTION_NAME)

Created collection: LearningArticle


In [5]:
article_data = [
    {
        "title": "What is Weaviate?",
        "content": (
            "Weaviate is an AI-native vector database that stores objects, metadata and vectors. "
            "It supports semantic search, keyword search, hybrid search and generative search."
        ),
        "source": "concept_notes",
    },
    {
        "title": "Local Docker Setup",
        "content": (
            "A local Weaviate instance can be started in Docker and used for development notebooks, "
            "experiments and local vector search testing."
        ),
        "source": "docker_notebook",
    },
    {
        "title": "Weaviate Cloud Connection",
        "content": (
            "Weaviate Cloud requires a cluster URL, a Weaviate API key and provider headers such as "
            "the OpenAI API key for text2vec_openai and generative_openai modules."
        ),
        "source": "cloud_notebook",
    },
    {
        "title": "Hybrid Search",
        "content": (
            "Hybrid search combines BM25 keyword search with vector search. "
            "The alpha parameter controls the balance between keyword-based and semantic ranking."
        ),
        "source": "search_notebook",
    },
    {
        "title": "Focused RAG",
        "content": (
            "Focused RAG works best when the collection contains clean chunks from relevant notebooks "
            "and Markdown notes instead of the entire unrelated project."
        ),
        "source": "rag_notebook",
    },
    {
        "title": "Multi-Tenancy",
        "content": (
            "Multi-tenancy allows one shared collection schema to isolate data for different companies, "
            "customers, users or organizations."
        ),
        "source": "saas_notebook",
    },
]

In [6]:
result = articles.data.insert_many(article_data)

if result.errors:
    print("Import errors:")
    print(result.errors)
else:
    print("Import completed.")

Import completed.


In [7]:
response = articles.query.fetch_objects(
    limit=20,
    return_properties=[
        "title",
        "content",
        "source",
    ],
)

for obj in response.objects:
    print("UUID:", obj.uuid)
    print("Title:", obj.properties["title"])
    print("Source:", obj.properties["source"])
    print("Content:", obj.properties["content"])
    print("-" * 80)

UUID: 0b97156c-0784-4708-833c-3e7953ce7873
Title: Focused RAG
Source: rag_notebook
Content: Focused RAG works best when the collection contains clean chunks from relevant notebooks and Markdown notes instead of the entire unrelated project.
--------------------------------------------------------------------------------
UUID: 199355fa-929c-46d3-b780-2eb56a9880a3
Title: Weaviate Cloud Connection
Source: cloud_notebook
Content: Weaviate Cloud requires a cluster URL, a Weaviate API key and provider headers such as the OpenAI API key for text2vec_openai and generative_openai modules.
--------------------------------------------------------------------------------
UUID: 2f3b785e-4d83-49bb-9e7a-4183af765cee
Title: Multi-Tenancy
Source: saas_notebook
Content: Multi-tenancy allows one shared collection schema to isolate data for different companies, customers, users or organizations.
--------------------------------------------------------------------------------
UUID: 47ccc779-4545-458f-a16c-

In [8]:
response = articles.query.near_text(
    query="how to build RAG with Weaviate notebooks",
    limit=3,
    return_properties=[
        "title",
        "content",
        "source",
    ],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print("Title:", obj.properties["title"])
    print("Source:", obj.properties["source"])
    print("Content:", obj.properties["content"])
    print("-" * 80)

Distance: 0.4250498414039612
Title: Local Docker Setup
Source: docker_notebook
Content: A local Weaviate instance can be started in Docker and used for development notebooks, experiments and local vector search testing.
--------------------------------------------------------------------------------
Distance: 0.44587457180023193
Title: Focused RAG
Source: rag_notebook
Content: Focused RAG works best when the collection contains clean chunks from relevant notebooks and Markdown notes instead of the entire unrelated project.
--------------------------------------------------------------------------------
Distance: 0.4903132915496826
Title: What is Weaviate?
Source: concept_notes
Content: Weaviate is an AI-native vector database that stores objects, metadata and vectors. It supports semantic search, keyword search, hybrid search and generative search.
--------------------------------------------------------------------------------


In [9]:
articles.config.add_property(
    wvc.config.Property(
        name="difficulty",
        data_type=wvc.config.DataType.TEXT,
    )
)

articles.config.add_property(
    wvc.config.Property(
        name="audience",
        data_type=wvc.config.DataType.TEXT,
    )
)

articles.config.add_property(
    wvc.config.Property(
        name="estimated_minutes",
        data_type=wvc.config.DataType.INT,
    )
)

articles.config.add_property(
    wvc.config.Property(
        name="reviewed",
        data_type=wvc.config.DataType.BOOL,
    )
)

print("New properties added.")

New properties added.


In [10]:
articles = client.collections.get(COLLECTION_NAME)

print("Collection reloaded:", COLLECTION_NAME)

Collection reloaded: LearningArticle


In [11]:
response = articles.query.fetch_objects(
    limit=20,
    return_properties=[
        "title",
        "source",
        "difficulty",
        "audience",
        "estimated_minutes",
        "reviewed",
    ],
)

for obj in response.objects:
    print("UUID:", obj.uuid)
    print("Title:", obj.properties["title"])
    print("Source:", obj.properties["source"])
    print("Difficulty:", obj.properties.get("difficulty"))
    print("Audience:", obj.properties.get("audience"))
    print("Estimated minutes:", obj.properties.get("estimated_minutes"))
    print("Reviewed:", obj.properties.get("reviewed"))
    print("-" * 80)

UUID: 0b97156c-0784-4708-833c-3e7953ce7873
Title: Focused RAG
Source: rag_notebook
Difficulty: None
Audience: None
Estimated minutes: None
Reviewed: None
--------------------------------------------------------------------------------
UUID: 199355fa-929c-46d3-b780-2eb56a9880a3
Title: Weaviate Cloud Connection
Source: cloud_notebook
Difficulty: None
Audience: None
Estimated minutes: None
Reviewed: None
--------------------------------------------------------------------------------
UUID: 2f3b785e-4d83-49bb-9e7a-4183af765cee
Title: Multi-Tenancy
Source: saas_notebook
Difficulty: None
Audience: None
Estimated minutes: None
Reviewed: None
--------------------------------------------------------------------------------
UUID: 47ccc779-4545-458f-a16c-4fa23393ee39
Title: What is Weaviate?
Source: concept_notes
Difficulty: None
Audience: None
Estimated minutes: None
Reviewed: None
--------------------------------------------------------------------------------
UUID: 65165915-3ebe-4d84-9a71-a1de

In [12]:
def infer_metadata(title: str, content: str) -> dict[str, object]:
    text = f"{title} {content}".lower()

    if "docker" in text:
        return {
            "difficulty": "beginner",
            "audience": "developer",
            "estimated_minutes": 15,
            "reviewed": True,
        }

    if "cloud" in text or "api key" in text:
        return {
            "difficulty": "beginner",
            "audience": "developer",
            "estimated_minutes": 20,
            "reviewed": True,
        }

    if "hybrid" in text or "bm25" in text:
        return {
            "difficulty": "intermediate",
            "audience": "developer",
            "estimated_minutes": 25,
            "reviewed": True,
        }

    if "rag" in text or "chunks" in text:
        return {
            "difficulty": "advanced",
            "audience": "ml_engineer",
            "estimated_minutes": 35,
            "reviewed": True,
        }

    if "multi-tenancy" in text or "customers" in text or "organizations" in text:
        return {
            "difficulty": "advanced",
            "audience": "saas_engineer",
            "estimated_minutes": 30,
            "reviewed": True,
        }

    return {
        "difficulty": "beginner",
        "audience": "general",
        "estimated_minutes": 10,
        "reviewed": False,
    }

In [13]:
response = articles.query.fetch_objects(
    limit=100,
    return_properties=[
        "title",
        "content",
        "source",
    ],
)

updated_count = 0

for obj in response.objects:
    metadata = infer_metadata(
        title=obj.properties["title"],
        content=obj.properties["content"],
    )

    articles.data.update(
        uuid=obj.uuid,
        properties=metadata,
    )

    updated_count += 1

print("Backfilled objects:", updated_count)

Backfilled objects: 6


In [14]:
response = articles.query.fetch_objects(
    limit=20,
    return_properties=[
        "title",
        "source",
        "difficulty",
        "audience",
        "estimated_minutes",
        "reviewed",
    ],
)

for obj in response.objects:
    print("UUID:", obj.uuid)
    print("Title:", obj.properties["title"])
    print("Source:", obj.properties["source"])
    print("Difficulty:", obj.properties["difficulty"])
    print("Audience:", obj.properties["audience"])
    print("Estimated minutes:", obj.properties["estimated_minutes"])
    print("Reviewed:", obj.properties["reviewed"])
    print("-" * 80)

UUID: 0b97156c-0784-4708-833c-3e7953ce7873
Title: Focused RAG
Source: rag_notebook
Difficulty: advanced
Audience: ml_engineer
Estimated minutes: 35
Reviewed: True
--------------------------------------------------------------------------------
UUID: 199355fa-929c-46d3-b780-2eb56a9880a3
Title: Weaviate Cloud Connection
Source: cloud_notebook
Difficulty: beginner
Audience: developer
Estimated minutes: 20
Reviewed: True
--------------------------------------------------------------------------------
UUID: 2f3b785e-4d83-49bb-9e7a-4183af765cee
Title: Multi-Tenancy
Source: saas_notebook
Difficulty: advanced
Audience: saas_engineer
Estimated minutes: 30
Reviewed: True
--------------------------------------------------------------------------------
UUID: 47ccc779-4545-458f-a16c-4fa23393ee39
Title: What is Weaviate?
Source: concept_notes
Difficulty: intermediate
Audience: developer
Estimated minutes: 25
Reviewed: True
-----------------------------------------------------------------------------

In [15]:
response = articles.query.fetch_objects(
    filters=Filter.by_property("difficulty").equal("advanced"),
    limit=20,
    return_properties=[
        "title",
        "source",
        "difficulty",
        "audience",
        "estimated_minutes",
    ],
)

for obj in response.objects:
    print("Title:", obj.properties["title"])
    print("Source:", obj.properties["source"])
    print("Difficulty:", obj.properties["difficulty"])
    print("Audience:", obj.properties["audience"])
    print("Estimated minutes:", obj.properties["estimated_minutes"])
    print("-" * 80)

Title: Focused RAG
Source: rag_notebook
Difficulty: advanced
Audience: ml_engineer
Estimated minutes: 35
--------------------------------------------------------------------------------
Title: Multi-Tenancy
Source: saas_notebook
Difficulty: advanced
Audience: saas_engineer
Estimated minutes: 30
--------------------------------------------------------------------------------


In [16]:
response = articles.query.fetch_objects(
    filters=Filter.by_property("audience").equal("developer"),
    limit=20,
    return_properties=[
        "title",
        "source",
        "difficulty",
        "audience",
    ],
)

for obj in response.objects:
    print("Title:", obj.properties["title"])
    print("Source:", obj.properties["source"])
    print("Difficulty:", obj.properties["difficulty"])
    print("Audience:", obj.properties["audience"])
    print("-" * 80)

Title: Weaviate Cloud Connection
Source: cloud_notebook
Difficulty: beginner
Audience: developer
--------------------------------------------------------------------------------
Title: What is Weaviate?
Source: concept_notes
Difficulty: intermediate
Audience: developer
--------------------------------------------------------------------------------
Title: Hybrid Search
Source: search_notebook
Difficulty: intermediate
Audience: developer
--------------------------------------------------------------------------------
Title: Local Docker Setup
Source: docker_notebook
Difficulty: beginner
Audience: developer
--------------------------------------------------------------------------------


In [17]:
response = articles.query.fetch_objects(
    filters=Filter.by_property("estimated_minutes").less_or_equal(25),
    limit=20,
    return_properties=[
        "title",
        "source",
        "difficulty",
        "estimated_minutes",
    ],
)

for obj in response.objects:
    print("Title:", obj.properties["title"])
    print("Source:", obj.properties["source"])
    print("Difficulty:", obj.properties["difficulty"])
    print("Estimated minutes:", obj.properties["estimated_minutes"])
    print("-" * 80)

Title: Weaviate Cloud Connection
Source: cloud_notebook
Difficulty: beginner
Estimated minutes: 20
--------------------------------------------------------------------------------
Title: What is Weaviate?
Source: concept_notes
Difficulty: intermediate
Estimated minutes: 25
--------------------------------------------------------------------------------
Title: Hybrid Search
Source: search_notebook
Difficulty: intermediate
Estimated minutes: 25
--------------------------------------------------------------------------------
Title: Local Docker Setup
Source: docker_notebook
Difficulty: beginner
Estimated minutes: 15
--------------------------------------------------------------------------------


In [18]:
response = articles.query.near_text(
    query="how to isolate data for SaaS customers",
    limit=5,
    return_properties=[
        "title",
        "content",
        "source",
        "difficulty",
        "audience",
        "estimated_minutes",
        "reviewed",
    ],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print("Title:", obj.properties["title"])
    print("Source:", obj.properties["source"])
    print("Difficulty:", obj.properties["difficulty"])
    print("Audience:", obj.properties["audience"])
    print("Estimated minutes:", obj.properties["estimated_minutes"])
    print("Reviewed:", obj.properties["reviewed"])
    print("Content:", obj.properties["content"])
    print("-" * 80)

Distance: 0.46158814430236816
Title: Multi-Tenancy
Source: saas_notebook
Difficulty: advanced
Audience: saas_engineer
Estimated minutes: 30
Reviewed: True
Content: Multi-tenancy allows one shared collection schema to isolate data for different companies, customers, users or organizations.
--------------------------------------------------------------------------------
Distance: 0.7072000503540039
Title: Weaviate Cloud Connection
Source: cloud_notebook
Difficulty: beginner
Audience: developer
Estimated minutes: 20
Reviewed: True
Content: Weaviate Cloud requires a cluster URL, a Weaviate API key and provider headers such as the OpenAI API key for text2vec_openai and generative_openai modules.
--------------------------------------------------------------------------------
Distance: 0.7630808353424072
Title: Local Docker Setup
Source: docker_notebook
Difficulty: beginner
Audience: developer
Estimated minutes: 15
Reviewed: True
Content: A local Weaviate instance can be started in Docker an

In [19]:
filters = (
    Filter.by_property("difficulty").equal("advanced")
    & Filter.by_property("reviewed").equal(True)
)

response = articles.query.near_text(
    query="production architecture with Weaviate",
    filters=filters,
    limit=5,
    return_properties=[
        "title",
        "content",
        "source",
        "difficulty",
        "audience",
        "estimated_minutes",
        "reviewed",
    ],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print("Title:", obj.properties["title"])
    print("Difficulty:", obj.properties["difficulty"])
    print("Audience:", obj.properties["audience"])
    print("Estimated minutes:", obj.properties["estimated_minutes"])
    print("Reviewed:", obj.properties["reviewed"])
    print("Content:", obj.properties["content"])
    print("-" * 80)

Distance: 0.721278727054596
Title: Multi-Tenancy
Difficulty: advanced
Audience: saas_engineer
Estimated minutes: 30
Reviewed: True
Content: Multi-tenancy allows one shared collection schema to isolate data for different companies, customers, users or organizations.
--------------------------------------------------------------------------------
Distance: 0.770463228225708
Title: Focused RAG
Difficulty: advanced
Audience: ml_engineer
Estimated minutes: 35
Reviewed: True
Content: Focused RAG works best when the collection contains clean chunks from relevant notebooks and Markdown notes instead of the entire unrelated project.
--------------------------------------------------------------------------------


In [20]:
response = articles.generate.near_text(
    query="advanced Weaviate features for real applications",
    filters=Filter.by_property("reviewed").equal(True),
    grouped_task=(
        "Answer using only the retrieved articles. "
        "Explain which articles are most useful for building real applications. "
        "Mention difficulty, audience and estimated reading time if available."
    ),
    limit=5,
    return_properties=[
        "title",
        "content",
        "source",
        "difficulty",
        "audience",
        "estimated_minutes",
        "reviewed",
    ],
)

print(response.generated)

print("\nSources:")
for obj in response.objects:
    print(
        "-",
        obj.properties["title"],
        "| difficulty:",
        obj.properties["difficulty"],
        "| audience:",
        obj.properties["audience"],
        "| minutes:",
        obj.properties["estimated_minutes"],
    )

Based on the retrieved articles, the following are the most useful for building real applications:

1. **What is Weaviate?**
   - **Audience:** Developer
   - **Difficulty:** Intermediate
   - **Estimated Reading Time:** 25 minutes
   - **Content Summary:** This article provides an overview of Weaviate, detailing its capabilities as an AI-native vector database. It covers semantic search, keyword search, hybrid search, and generative search, which are crucial for understanding how to leverage Weaviate in real applications.

2. **Weaviate Cloud Connection**
   - **Audience:** Developer
   - **Difficulty:** Beginner
   - **Estimated Reading Time:** 20 minutes
   - **Content Summary:** This article explains how to connect to Weaviate Cloud, requiring detailed information about cluster URLs and API keys. It is essential for developers looking to utilize Weaviate's cloud capabilities in their applications.

3. **Local Docker Setup**
   - **Audience:** Developer
   - **Difficulty:** Beginner

In [21]:
def recommend_article(
    query: str,
    *,
    difficulty: str | None = None,
    max_minutes: int | None = None,
    audience: str | None = None,
    limit: int = 5,
) -> None:
    filters = Filter.by_property("reviewed").equal(True)

    if difficulty is not None:
        filters = filters & Filter.by_property("difficulty").equal(difficulty)

    if max_minutes is not None:
        filters = filters & Filter.by_property("estimated_minutes").less_or_equal(max_minutes)

    if audience is not None:
        filters = filters & Filter.by_property("audience").equal(audience)

    response = articles.query.near_text(
        query=query,
        filters=filters,
        limit=limit,
        return_properties=[
            "title",
            "content",
            "source",
            "difficulty",
            "audience",
            "estimated_minutes",
        ],
        return_metadata=MetadataQuery(distance=True),
    )

    print("QUERY:", query)
    print("DIFFICULTY:", difficulty)
    print("MAX MINUTES:", max_minutes)
    print("AUDIENCE:", audience)
    print("-" * 80)

    for obj in response.objects:
        print("Distance:", obj.metadata.distance)
        print("Title:", obj.properties["title"])
        print("Source:", obj.properties["source"])
        print("Difficulty:", obj.properties["difficulty"])
        print("Audience:", obj.properties["audience"])
        print("Estimated minutes:", obj.properties["estimated_minutes"])
        print("Content:", obj.properties["content"])
        print("-" * 80)

In [22]:
recommend_article(
    "I want to understand Weaviate Cloud authentication",
    difficulty="beginner",
    max_minutes=25,
    audience="developer",
)

QUERY: I want to understand Weaviate Cloud authentication
DIFFICULTY: beginner
MAX MINUTES: 25
AUDIENCE: developer
--------------------------------------------------------------------------------
Distance: 0.3800284266471863
Title: Weaviate Cloud Connection
Source: cloud_notebook
Difficulty: beginner
Audience: developer
Estimated minutes: 20
Content: Weaviate Cloud requires a cluster URL, a Weaviate API key and provider headers such as the OpenAI API key for text2vec_openai and generative_openai modules.
--------------------------------------------------------------------------------
Distance: 0.5709404349327087
Title: Local Docker Setup
Source: docker_notebook
Difficulty: beginner
Audience: developer
Estimated minutes: 15
Content: A local Weaviate instance can be started in Docker and used for development notebooks, experiments and local vector search testing.
--------------------------------------------------------------------------------


In [23]:
recommend_article(
    "I want to build RAG with notebooks and markdown notes",
    difficulty="advanced",
    max_minutes=40,
)

QUERY: I want to build RAG with notebooks and markdown notes
DIFFICULTY: advanced
MAX MINUTES: 40
AUDIENCE: None
--------------------------------------------------------------------------------
Distance: 0.35351550579071045
Title: Focused RAG
Source: rag_notebook
Difficulty: advanced
Audience: ml_engineer
Estimated minutes: 35
Content: Focused RAG works best when the collection contains clean chunks from relevant notebooks and Markdown notes instead of the entire unrelated project.
--------------------------------------------------------------------------------
Distance: 0.7611287832260132
Title: Multi-Tenancy
Source: saas_notebook
Difficulty: advanced
Audience: saas_engineer
Estimated minutes: 30
Content: Multi-tenancy allows one shared collection schema to isolate data for different companies, customers, users or organizations.
--------------------------------------------------------------------------------


In [24]:
recommend_article(
    "I am building SaaS with isolated customer data",
    difficulty="advanced",
    max_minutes=35,
    audience="saas_engineer",
)

QUERY: I am building SaaS with isolated customer data
DIFFICULTY: advanced
MAX MINUTES: 35
AUDIENCE: saas_engineer
--------------------------------------------------------------------------------
Distance: 0.4694553017616272
Title: Multi-Tenancy
Source: saas_notebook
Difficulty: advanced
Audience: saas_engineer
Estimated minutes: 30
Content: Multi-tenancy allows one shared collection schema to isolate data for different companies, customers, users or organizations.
--------------------------------------------------------------------------------


In [25]:
client.close()